<a href="https://colab.research.google.com/github/AnNhiAI/A.N.T.H.O.R/blob/master/01_tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Tokenizer

Train a **Byte-Level BPE (Byte Pair Encoding)** tokenizer for A.N.T.H.O.R.
Saved directly to **Google Drive**.

> Runs on **Google Colab** (free T4 tier).

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers datasets transformers

## 2. Mount Google Drive (MANDATORY)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

In [ ]:
# ─── Tokenizer config ───
VOCAB_SIZE   = 16_384
MIN_FREQUENCY = 2

# ─── Data config ───
DATASETS = [
    {
        "name": "HuggingFaceFW/fineweb-edu",
        "config": "sample-10BT",
        "split": "train",
        "sample_size": 10000,
        "text_key": "text"
    },
    {
        "name": "hirine/wikipedia-vietnamese-1M296K-dataset",
        "config": None,
        "split": "train",
        "sample_size": 10000,
        "text_key": "text"
    }
]

# ─── Google Drive paths (MANDATORY) ───
DRIVE_BASE     = "/content/drive/MyDrive/ANTHOR"
TOKENIZER_PATH = f"{DRIVE_BASE}/tokenizer/tokenizer.json"

import os
os.makedirs(f"{DRIVE_BASE}/tokenizer", exist_ok=True)

## 4. Load dataset

In [ ]:
from datasets import load_dataset
import random

docs = []
for ds_info in DATASETS:
    name = ds_info["name"]
    cfg = ds_info.get("config")
    split = ds_info.get("split", "train")
    size = ds_info.get("sample_size", 5000)
    key = ds_info.get("text_key", "text")
    print(f"Loading {name} ({cfg or 'default'}/{split})...")
    try:
        ds = load_dataset(name, cfg, split=split, streaming=True)
        count = 0
        for example in ds:
            if count >= size:
                break
            text = example.get(key)
            if text:
                docs.append(text)
                count += 1
                if count % 2000 == 0:
                    print(f"  Loaded {count}/{size}")
    except Exception as e:
        print(f"Error loading {name}: {e}")

random.shuffle(docs)
total_bytes = sum(len(d.encode("utf-8")) for d in docs)
print(f"\nLoaded {len(docs)} docs in total, ~{total_bytes / 1e6:.1f} MB")

## 5. Train & Save Tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
import time

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

print(f"Training BPE (vocab_size={VOCAB_SIZE})...")
t0 = time.time()
tokenizer.train_from_iterator(docs, trainer)
print(f"Done in {time.time() - t0:.1f}s, vocab_size={tokenizer.get_vocab_size()}")

# ── Save to Google Drive (MANDATORY) ──
os.makedirs(os.path.dirname(TOKENIZER_PATH), exist_ok=True)
tokenizer.save(TOKENIZER_PATH)
print(f"Saved to Drive: {TOKENIZER_PATH}")

## 6. Verify saved tokenizer

In [ ]:
import os
loaded = Tokenizer.from_file(TOKENIZER_PATH)
print(f"Loaded tokenizer from Drive: vocab_size={loaded.get_vocab_size()}")
test = loaded.encode("Xin chào Việt Nam!")
print(f"Test encode: {test.ids}")
print(f"Test decode: {loaded.decode(test.ids)}")

---

✅ **Next:** `02_data_pipeline.ipynb` (saves to Google Drive)

> **Note:** All data/tokenizer/models are now **Google Drive only** — no local storage.